In [1]:
import pandas as pd
import numpy as np
import yaml
from sqlalchemy import create_engine

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# connecting to tha database for later use:
#------------------------------------------
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

connection = create_engine(config['bank_data_db'])


# spliting the raw data into 3 tables:
#-------------------------------------
raw_data = pd.read_csv('../data/raw/bank.csv')

client_original = raw_data[['client_id', 'age', 'job', 'marital', 'education', 'credit_default', 'housing', 'loan']]
campaign_original = raw_data[['client_id', 'campaign', 'duration', 'pdays', 'previous', 'poutcome', 'y', 'month', 'day']]
economics_original = raw_data[['client_id', 'emp_var_rate', 'cons_price_idx', 'euribor3m', 'nr_employed']]

client = client_original.copy()
campaign = campaign_original.copy()
economics = economics_original.copy()


# cleaning the `client` table:
#-----------------------------
client.rename(columns={'client_id':'id'}, inplace=True)
client['credit_default'] = client['credit_default'].replace({'yes':1, 'no':0, 'unknown':np.nan}).astype(bool)
client['housing'] = client['housing'].replace({'yes':1, 'no':0, 'unknown':np.nan}).fillna(0).astype(bool)
client['loan'] = client['loan'].replace({'yes':1, 'no':0, 'unknown':np.nan}).fillna(0).astype(bool)
client['education'] = client['education'].str.replace('.', '_').replace({'unknown':np.nan})


# cleaning the `campaign` table:
#-------------------------------
campaign.insert(0, 'campaign_id',range(len(campaign)))
campaign['poutcome'] = campaign['poutcome'].replace({'nonexistent':np.nan}).fillna(0).astype(bool)
campaign['y'] = campaign['y'].replace({'yes':1, 'no':0}).astype(bool)
campaign['month'] = campaign['month'].str.capitalize()
campaign['day'] = campaign['day'].astype(str)
campaign['year'] = '2022'
campaign['last_contact_date'] = campaign['year']+'-'+campaign['month']+'-'+campaign['day']
campaign['last_contact_date'] = pd.to_datetime(campaign['last_contact_date'])
campaign.drop(columns=['year', 'day', 'month'], inplace=True)
campaign.rename(columns={'campaign':'number_contacts', 'duration':'contact_duration', 'previous':'previous_campaign_contacts', 'poutcome':'previous_outcome', 'y':'campaign_outcome'}, inplace=True)


# cleaning the `economics` table:
#--------------------------------
economics['nr_employed'] = economics['nr_employed'].astype(float)
economics.rename(columns={'euribor3m':'euribor_three_months', 'nr_employed':'number_employed'}, inplace=True)


# saving all the 3 tables as CSV files:
#--------------------------------------
def save_files():
    client.to_csv('../data/processed/client.csv', index=False)
    campaign.to_csv('../data/processed/campaign.csv', index=False)
    economics.to_csv('../data/processed/economics.csv', index=False)


# loading all the 3 tables to the database:
#------------------------------------------
def load_to_database():
    client.to_sql(con=connection, name='client', if_exists='fail', index=False)
    campaign.to_sql(con=connection, name='campaign', if_exists='fail', index=False)
    economics.to_sql(con=connection, name='economics', if_exists='fail', index=False)
